<a href="https://colab.research.google.com/github/russianoracle/punct-nlu-colab-training/blob/main/PunctNLU_Colab_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PunctNLU CUDA Training on Colab
Training Russian punctuation restoration model on GPU

**Model:** rubert-tiny2 (29.2M params) → 12-class punctuation + capitalization

**Corpus:** 1.194M Russian sentences from Tatoeba

**Hardware:** Colab GPU (T4/A100)

## Step 1: Setup & Install Dependencies

In [25]:
!pip install -q torch transformers numpy pandas scikit-learn tqdm

In [26]:
import json
import logging
import re
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

# Check GPU
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU Available: True
GPU Name: Tesla T4
GPU Memory: 15.6 GB


from google.colab import drive

# Optional: Mount Drive only if you need to save results there
# drive.mount('/content/drive', force_remount=True)
print("✓ Colab environment ready")

In [ ]:
from google.colab import drive

# Mount Google Drive to access corpus
print("📂 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

# Corpus location on Google Drive
# Upload sentences.csv to: My Drive/PunctNLU/sentences.csv
CORPUS_PATH = Path("/content/drive/MyDrive/PunctNLU/sentences.csv")

if CORPUS_PATH.exists():
    size_mb = CORPUS_PATH.stat().st_size / 1e6
    print(f"✓ Corpus ready: {size_mb:.0f} MB")
else:
    print("✗ Corpus not found at /content/drive/MyDrive/PunctNLU/sentences.csv")
    print("  Please upload sentences.csv to Google Drive first")

### ☝️ Before running: Upload sentences.csv to Google Drive
1. Download sentences.csv from: https://github.com/russianoracle/AIssistant/releases/download/v1.0-assets/sentences.csv
2. Upload to Google Drive → My Drive → Create folder "PunctNLU" → Upload sentences.csv there
3. Run the cells above — Colab will mount Drive and find the corpus automatically
✅ Then training starts!

## Step 3: Configuration (CUDA-Optimized)

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────

MODEL_ID      = "cointegrated/rubert-tiny2"
SEQ_LEN       = 64
NUM_LABELS    = 12
BATCH_SIZE    = 32  # Higher on GPU (was 16 on M2)
NUM_EPOCHS    = 3
LEARNING_RATE = 1e-4
WARMUP_STEPS  = 500
WEIGHT_DECAY  = 1e-5
NUM_WORKERS   = 2
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR    = Path("./checkpoints/punct_nlu")

PUNCT_CHARS   = {",": 1, ".": 2, "!": 2, "…": 2, ";": 2, "?": 3}
# CORPUS_PATH is already set in cell 6 (mounted from Google Drive)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

log.info(f"Device: {DEVICE}")
log.info(f"Batch Size: {BATCH_SIZE}")
log.info(f"Model: {MODEL_ID}")
log.info(f"Corpus: {CORPUS_PATH}")

## Step 4: Data Loading & Processing

In [ ]:
# ── Label extraction ───────────────────────────────────────────────────────
from typing import Optional

def cap_mode_vec(words: np.ndarray) -> np.ndarray:
    """Vectorized capitalization detection (faster than list comp)"""
    modes = np.zeros(len(words), dtype=np.int32)
    for i, word in enumerate(words):
        clean = re.sub(r"[^\w]", "", word, flags=re.UNICODE)
        if clean:
            alpha = [c for c in clean if c.isalpha()]
            if alpha:
                if all(c.isupper() for c in alpha):
                    modes[i] = 2
                elif alpha[0].isupper():
                    modes[i] = 1
    return modes


def punct_class_vec(words: np.ndarray) -> np.ndarray:
    """Vectorized punctuation detection (faster than list comp)"""
    classes = np.zeros(len(words), dtype=np.int32)
    for i, word in enumerate(words):
        if word:
            last = word[-1]
            classes[i] = PUNCT_CHARS.get(last, 0)
    return classes


def word_labels_vec(words: list[str]) -> np.ndarray:
    """Vectorized label creation from word list"""
    words_arr = np.array(words, dtype=object)
    punct_classes = punct_class_vec(words_arr)
    cap_modes = cap_mode_vec(words_arr)
    return punct_classes * 3 + cap_modes


def ortho_to_norm(word: str) -> str:
    return re.sub(r"[^\w]", "", word, flags=re.UNICODE).lower()

In [ ]:
# ── Data loader ────────────────────────────────────────────────────────────

def iter_tatoeba(path: Path) -> Iterator[tuple[list[str], list[int]]]:
    """Read Tatoeba corpus without limits"""
    count = 0
    with path.open(encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t", 2)
            if len(parts) != 3 or parts[1] != "rus":
                continue

            sentence = parts[2].strip()
            if len(sentence) < 5:
                continue

            words = sentence.split()
            if len(words) < 2:
                continue

            labels = word_labels_vec(words).tolist()
            norms = [ortho_to_norm(w) for w in words]

            if not any(norms):
                continue

            yield norms, labels
            count += 1
            if count % 100000 == 0:
                log.info(f"  Loaded {count} sentences...")


@dataclass
class EncodedSample:
    input_ids: torch.Tensor      # [SEQ_LEN]
    attention_mask: torch.Tensor # [SEQ_LEN]
    labels: torch.Tensor         # [SEQ_LEN]


def encode(norm_words: list[str], word_labels: list[int], tokenizer) -> Optional[EncodedSample]:
    """Encode sentence with proper word-start alignment"""
    ids = np.zeros(SEQ_LEN, dtype=np.int32)
    mask = np.zeros(SEQ_LEN, dtype=np.int32)
    labels = np.full(SEQ_LEN, -100, dtype=np.int32)

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id

    ids[0] = cls_id
    mask[0] = 1
    pos = 1

    for word, lbl in zip(norm_words, word_labels):
        if pos >= SEQ_LEN - 1:
            break

        sub_ids = tokenizer.encode(word, add_special_tokens=False)
        if not sub_ids:
            sub_ids = [tokenizer.unk_token_id]

        if pos + len(sub_ids) >= SEQ_LEN:
            break

        ids[pos] = sub_ids[0]
        mask[pos] = 1
        labels[pos] = lbl
        pos += 1

        for sub_id in sub_ids[1:]:
            if pos >= SEQ_LEN:
                break
            ids[pos] = sub_id
            mask[pos] = 1
            labels[pos] = -100
            pos += 1

    if pos < SEQ_LEN:
        ids[pos] = sep_id
        mask[pos] = 1
        pos += 1

    return EncodedSample(
        input_ids=torch.from_numpy(ids).long(),
        attention_mask=torch.from_numpy(mask).long(),
        labels=torch.from_numpy(labels).long(),
    )


class PunctDataset(Dataset):
    def __init__(self, samples: list[EncodedSample]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            "input_ids": s.input_ids,
            "attention_mask": s.attention_mask,
            "labels": s.labels,
        }

## Step 5: Model Definition

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────

class PunctNLUModel(nn.Module):
    """Two-head model: punct restoration + actionable classification"""

    def __init__(self, model_id: str, num_labels: int = 12):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_id)
        self.dropout = nn.Dropout(0.1)

        # Head A1: Punctuation restoration (12 classes)
        self.punct_head = nn.Linear(self.bert.config.hidden_size, num_labels)

        # Head A2: Actionable classification (2 classes)
        self.classify_head = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state  # [B, L, H]
        hidden = self.dropout(hidden)

        punct_logits = self.punct_head(hidden)      # [B, L, 12]
        classify_logits = self.classify_head(hidden) # [B, L, 2]

        return punct_logits, classify_logits

## Step 6: Training Loop

In [ ]:
# ── Training ───────────────────────────────────────────────────────────────

def train_epoch(model, loader, optimizer, scheduler, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0.0

    ce_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    for batch_idx, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        punct_logits, classify_logits = model(input_ids, attention_mask)

        loss_punct = ce_loss_fn(punct_logits.view(-1, 12), labels.view(-1))
        loss_classify = ce_loss_fn(
            classify_logits.view(-1, 2),
            (labels >= 0).long().view(-1) * labels.view(-1).clamp(min=0)
        )

        loss = loss_punct + 0.5 * loss_classify
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 100 == 0:
            log.info(f"  Batch {batch_idx + 1}/{len(loader)}: loss={loss.item():.4f}")

    return total_loss / len(loader)


def eval_epoch(model, loader, device):
    """Evaluate for one epoch"""
    model.eval()
    total_loss = 0.0
    correct = torch.tensor(0, device=device)
    total = torch.tensor(0, device=device)

    ce_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            punct_logits, classify_logits = model(input_ids, attention_mask)

            loss_punct = ce_loss_fn(punct_logits.view(-1, 12), labels.view(-1))
            loss_classify = ce_loss_fn(
                classify_logits.view(-1, 2),
                (labels >= 0).long().view(-1) * labels.view(-1).clamp(min=0)
            )

            loss = loss_punct + 0.5 * loss_classify
            total_loss += loss.item()

            pred = torch.argmax(punct_logits, dim=-1)
            mask = labels >= 0
            correct += (pred[mask] == labels[mask]).sum()
            total += mask.sum()

    return total_loss / len(loader), (correct / max(total, 1)).item()

## Step 7: Main Training Function

In [ ]:
def main():
    log.info(f"Device: {DEVICE}")

    # ── 1. Load corpus ──────────────────────────────────────────────────────
    log.info(f"Loading corpus from {CORPUS_PATH}...")
    if not CORPUS_PATH.exists():
        log.error(f"Corpus not found: {CORPUS_PATH}")
        return False

    corpus_data = list(iter_tatoeba(CORPUS_PATH))
    log.info(f"Loaded {len(corpus_data)} samples")

    # ── 2. Load tokenizer ───────────────────────────────────────────────────
    log.info(f"Loading tokenizer: {MODEL_ID}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

    # ── 3. Encode all samples ───────────────────────────────────────────────
    log.info("Encoding samples...")
    encoded_samples = []
    for norm_words, labels in corpus_data:
        sample = encode(norm_words, labels, tokenizer)
        if sample:
            encoded_samples.append(sample)

    log.info(f"Encoded {len(encoded_samples)} samples")

    # ── 4. Split train/val ──────────────────────────────────────────────────
    np.random.seed(42)
    indices = np.random.permutation(len(encoded_samples))
    split = int(0.95 * len(encoded_samples))

    train_indices = indices[:split]
    val_indices = indices[split:]

    train_dataset = PunctDataset([encoded_samples[i] for i in train_indices])
    val_dataset = PunctDataset([encoded_samples[i] for i in val_indices])

    log.info(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,  # Colab: set to 0
        pin_memory=False,  # GPU doesn't need pin_memory
        persistent_workers=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        num_workers=0,  # Colab: set to 0
        pin_memory=False,
        persistent_workers=False,
    )

    # ── 5. Initialize model ─────────────────────────────────────────────────
    log.info("Building model...")
    model = PunctNLUModel(MODEL_ID).to(DEVICE)
    log.info(f"Params: {sum(p.numel() for p in model.parameters()):,} trainable / "
             f"{sum(p.numel() for p in model.parameters()):,} total")

    # ── 6. Training loop ────────────────────────────────────────────────────
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    total_steps = len(train_loader) * NUM_EPOCHS
    scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=1.0,
        end_factor=0.0,
        total_iters=total_steps,
    )

    best_val_loss = float("inf")

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()

        train_loss = train_epoch(model, train_loader, optimizer, scheduler, DEVICE)
        val_loss, val_acc = eval_epoch(model, val_loader, DEVICE)

        elapsed = time.time() - t0
        log.info(f"Epoch {epoch}/{NUM_EPOCHS}  train={train_loss:.4f}  "
                f"val={val_loss:.4f}  acc={val_acc:.3f}  t={elapsed:.0f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), OUTPUT_DIR / "best.pt")
            log.info(f"  ✓ saved best → {OUTPUT_DIR / 'best.pt'}")

    log.info(f"Training done. Best val loss: {best_val_loss:.4f}")

    # ── Save final checkpoint ───────────────────────────────────────────────
    torch.save({
        "model_state_dict": model.state_dict(),
        "best_val_loss": best_val_loss,
    }, OUTPUT_DIR / "punct_nlu.pt")
    log.info(f"Saved final checkpoint → {OUTPUT_DIR / 'punct_nlu.pt'}")

    return True

## Step 8: RUN TRAINING

In [ ]:
success = main()
if success:
    print("\n✓ Training completed successfully!")
else:
    print("\n✗ Training failed. Check corpus file location.")

## Step 9: Download Results

In [ ]:
from google.colab import files

# Download trained model
print("Downloading trained model...")
files.download("checkpoints/punct_nlu/best.pt")
print("✓ Downloaded: best.pt")